# Sparse Retrieval (BM25): Practical Demo

In this notebook we demonstrate **Sparse Retrieval (BM25)** on a multi-topic text using a real LLM (`ChatGroq`).

**What is it?** Instead of converting text into dense embeddings, sparse retrieval matches the **exact words** of the query with the words in each chunk. No embeddings, no GPU.

**BM25** rewards **rare words** over common words using Inverse Document Frequency (IDF).

**What we will do**
1. Setup & connect the LLM (`ChatGroq`)
2. Load text, split into chunks, and build the BM25 index
3. Helper functions (ranking, word IDF weights, RAG generation)
4. Exact keyword & technical term retrieval (advantage)
5. Paraphrasing & synonym limitation (disadvantage)

## 1. Setup

### 1.1 Install packages and connect the LLM

In [1]:
pip install langchain-groq rank_bm25 langchain-text-splitters -q # q for silent installation

In [2]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

## 2. Load text, chunk it, and build the BM25 index

### 2.1 Define the text and split into chunks

Our text covers three distinct topics (**Photosynthesis**, **The French Revolution**, and **Machine Learning**). We split it using `RecursiveCharacterTextSplitter`.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
Chlorophyll in the chloroplasts absorbs the light.

The French Revolution began in 1789 and fundamentally changed European society.
Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.

Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
Transformers have become the dominant architecture since 2017.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=160, chunk_overlap=20)
chunks = [c.strip() for c in splitter.split_text(text) if c.strip()]

print('Total chunks:', len(chunks))
for i, chunk in enumerate(chunks):
    print(f'{i} | {chunk}')

Total chunks: 6
0 | Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
1 | Chlorophyll in the chloroplasts absorbs the light.
2 | The French Revolution began in 1789 and fundamentally changed European society.
3 | Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.
4 | Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
5 | Transformers have become the dominant architecture since 2017.


### 2.2 Build the BM25 index

We lowercase and extract exact word tokens (no stemming or synonyms), then index them with `BM25Okapi`.

In [5]:
import re
import time
from rank_bm25 import BM25Okapi

def tokenize(s):
    # lowercase and split into words
    return re.findall(r'\w+', s.lower())

start = time.time()
tokenized_chunks = [tokenize(chunk) for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f'Index built for {len(chunks)} chunks in {time.time() - start:.4f} seconds')

Index built for 6 chunks in 0.0012 seconds


### 2.3 Helper functions

- `show_top`: displays the top-k highest-scoring chunks.
- `word_weights`: shows how many chunks contain each query word and its BM25 IDF weight.
- `ask`: runs grounded RAG generation using `ChatGroq`.

In [6]:
import numpy as np
import pandas as pd

def show_top(scores, k=3):
    # Indices of the k chunks with the highest scores
    top_ids = np.argsort(scores)[::-1][:k]
    for rank, i in enumerate(top_ids, start=1):
        print(f'{rank}. score = {scores[i]:.2f} | chunk[{i}]: {chunks[i]}')
    return top_ids

def word_weights(query_tokens):
    # For every query word: in how many chunks does it appear, and how much does BM25 value it (idf)?
    rows = []
    for word in query_tokens:
        rows.append({
            'word': word,
            'chunks_containing_word': sum(word in set(tokens) for tokens in tokenized_chunks),
            'bm25_weight (idf)': round(bm25.idf.get(word, 0), 2)
        })
    return pd.DataFrame(rows).sort_values('bm25_weight (idf)', ascending=False)

def ask(question, retrieved_chunks):
    # Augmentation: combine the retrieved chunks with the user question
    context = '\n\n'.join(retrieved_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""

    # Generation: LLM answers on the basis of the retrieved info
    return llm.invoke(prompt).content

## 3. Advantage: exact names and technical terms

When the query uses exact keywords present in the text (such as `chloroplasts`, `photosynthesis`, or `louis`), BM25 assigns high IDF weights to those rare domain terms and retrieves the exact right chunk.

In [7]:
query = 'What absorbs light in chloroplasts during photosynthesis?'
query_tokens = tokenize(query)

word_weights(query_tokens)

,word,chunks_containing_word,bm25_weight (idf)
1,absorbs,1,1.30
6,photosynthesis,1,1.30
4,chloroplasts,1,1.30
2,light,2,0.59
0,what,0,0.00
3,in,3,0.00
5,during,0,0.00


In [8]:
bm25_scores = bm25.get_scores(query_tokens)
top_ids = show_top(bm25_scores, k=2)

print()
print('LLM answer:', ask(query, [chunks[i] for i in top_ids]))

1. score = 4.15 | chunk[1]: Chlorophyll in the chloroplasts absorbs the light.
2. score = 1.66 | chunk[0]: Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.

LLM answer: Chlorophyll in the chloroplasts absorbs light during photosynthesis.


### Example 2: Historical proper noun (`Louis XVI`)

In [9]:
query = 'When was Louis XVI executed?'
query_tokens = tokenize(query)

display(word_weights(query_tokens))

bm25_scores = bm25.get_scores(query_tokens)
top_ids = show_top(bm25_scores, k=2)

print()
print('LLM answer:', ask(query, [chunks[i] for i in top_ids]))

,word,chunks_containing_word,bm25_weight (idf)
1,was,1,1.3
3,xvi,1,1.3
2,louis,1,1.3
4,executed,1,1.3
0,when,0,0.0


1. score = 3.92 | chunk[3]: Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.
2. score = 0.00 | chunk[5]: Transformers have become the dominant architecture since 2017.

LLM answer: 1793


## 4. Disadvantage: no synonyms or paraphrasing

BM25 has **no concept of semantic meaning**. If a query uses synonyms or paraphrases (e.g., *"artificial neural nets"* instead of *"Neural networks"*, or *"monarch beheaded"* instead of *"Louis XVI was executed"*), those words get `0` matches.

In [10]:
query = 'Which French monarch was beheaded in 1793?'
query_tokens = tokenize(query)

display(word_weights(query_tokens))

bm25_scores = bm25.get_scores(query_tokens)
top_ids = show_top(bm25_scores, k=2)

print()
print('LLM answer:', ask(query, [chunks[i] for i in top_ids]))

,word,chunks_containing_word,bm25_weight (idf)
1,french,1,1.3
6,1793,1,1.3
3,was,1,1.3
2,monarch,0,0.0
0,which,0,0.0
4,beheaded,0,0.0
5,in,3,0.0


1. score = 1.96 | chunk[3]: Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.
2. score = 1.46 | chunk[2]: The French Revolution began in 1789 and fundamentally changed European society.

LLM answer: Louis XVI.


In [11]:
query = 'How do artificial neural nets train with gradient descent?'
query_tokens = tokenize(query)

display(word_weights(query_tokens))

bm25_scores = bm25.get_scores(query_tokens)
top_ids = show_top(bm25_scores, k=2)

print()
print('LLM answer:', ask(query, [chunks[i] for i in top_ids]))

,word,chunks_containing_word,bm25_weight (idf)
2,artificial,1,1.3
7,gradient,1,1.3
3,neural,1,1.3
8,descent,1,1.3
0,how,0,0.0
4,nets,0,0.0
1,do,0,0.0
6,with,0,0.0
5,train,0,0.0


1. score = 4.82 | chunk[4]: Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
2. score = 0.00 | chunk[5]: Transformers have become the dominant architecture since 2017.

LLM answer: Neural networks learn patterns from data through gradient descent.
